# Lecture 1: the Solow model on a computer

**ECU44291 Quantitative Macroeconomics, Week 1.**

Three things a computer does with a model: **evaluate a function**, **iterate a loop**, **find a fixed point**. Today we do all three on a model you already know, so that next week we can do them on one you cannot solve by hand.

In efficiency units, with $k_t = K_t/(A_t L_t)$:

$$k_{t+1} = g(k_t) = s\,k_t^{\alpha} + (1-\delta-n-g)\,k_t, \qquad k^* = \left(\frac{s}{\delta+n+g}\right)^{1/(1-\alpha)}.$$

Parameters for today: $\alpha=0.33$, $s=0.30$, $n=0.03$, $g=0.04$, $\delta=0.15$.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Task 1: the Solow equation as a Python function

Write `k_next(k, ...)` returning $k_{t+1}$ for a given $k_t$, and `steady_state(...)` returning $k^*$. Give the parameters default values so both can be called with no arguments. Write a one-line docstring for each.

In [ ]:
# SOLUTION
def k_next(k, alpha=0.33, s=0.30, n=0.03, g=0.04, delta=0.15):
    """Next period's capital per effective worker."""
    return s * k**alpha + (1 - delta - n - g) * k


def steady_state(alpha=0.33, s=0.30, n=0.03, g=0.04, delta=0.15):
    """Closed-form steady state k*."""
    return (s / (delta + n + g)) ** (1 / (1 - alpha))


kstar = steady_state()
print(f"k* = {kstar:.4f}")
print(f"g(k*) - k* = {k_next(kstar) - kstar:.2e}")   # a fixed point: should be ~0

## Task 2: the 45-degree diagram

Plot $g(k)$ for $k \in [0, 4]$ together with the 45-degree line and mark $k^*$. Where the two lines cross is the fixed point. Where $g$ is above the line, $k$ grows; below it, $k$ shrinks.

In [ ]:
# SOLUTION
k = np.linspace(0, 4, 400)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(k, k, "--", color="grey", label="45-degree line")
ax.plot(k, k_next(k), label="$g(k)$")
ax.plot(kstar, kstar, "o", label=f"$k^* = {kstar:.2f}$")
ax.set_xlabel("$k_t$"); ax.set_ylabel("$k_{t+1}$"); ax.legend()
plt.show()

## Task 3: iterate

Write `simulate(k0, T, **params)` returning the path $k_0, k_1, \dots, k_T$ as a NumPy array. Use a loop; we will vectorise things later when there is a reason to.

In [ ]:
# SOLUTION
def simulate(k0, T, **params):
    """Path of k from k0 for T periods (length T+1)."""
    path = [k0]
    for t in range(T):
        path.append(k_next(path[-1], **params))
    return np.array(path)


print(simulate(0.25, 5))

## Task 4: time paths

Plot the paths from $k_0 \in \{0.25, 1.25, 3.5\}$ for 25 periods on one figure, with a horizontal line at $k^*$. Label each path.

In [ ]:
# SOLUTION
T = 25
fig, ax = plt.subplots(figsize=(7, 4))
for k0 in (0.25, 1.25, 3.5):
    ax.plot(simulate(k0, T), label=f"$k_0 = {k0}$")
ax.axhline(kstar, ls=":", color="grey")
ax.set_xlabel("period"); ax.set_ylabel("$k_t$"); ax.legend()
plt.show()

## Task 5: the fixed point, without the formula

Pretend the closed form for $k^*$ did not exist. Write `fixed_point(g, x0, tol=1e-8, max_iter=1000)` that repeats $x \leftarrow g(x)$ until $|g(x) - x| < \text{tol}$, returns the fixed point and the number of iterations, and **raises an error** if `max_iter` is reached. Check it against the closed form.

Every algorithm in this module has the same three parts: a loop, a stopping rule, and an update. Point at each one in your code.

In [ ]:
# SOLUTION
def fixed_point(g, x0, tol=1e-8, max_iter=1000):
    """Iterate x <- g(x) from x0 until the change is below tol."""
    x = x0
    for it in range(max_iter):          # the loop
        x_new = g(x)                    # the update
        if abs(x_new - x) < tol:        # the stopping rule
            return x_new, it + 1
        x = x_new
    raise RuntimeError(f"no convergence after {max_iter} iterations")


x, n_it = fixed_point(k_next, 0.25)
print(f"fixed point {x:.8f} after {n_it} iterations; closed form {kstar:.8f}")

## Task 6: how fast, and why

Near $k^*$, $k_{t+1} - k^* \approx g'(k^*)(k_t - k^*)$ with $g'(k^*) = 1 - (1-\alpha)(\delta+n+g)$.

1. Compute $g'(k^*)$ from the formula and numerically (a finite difference). Do they agree?
2. Plot $|k_t - k^*|$ on a log scale for the three starting points. Why is it a straight line? What is its slope?
3. Change $\delta$ to $0.02$. What happens to the number of iterations `fixed_point` needs, and why?

In [ ]:
# SOLUTION
alpha, s, n, g, delta = 0.33, 0.30, 0.03, 0.04, 0.15
slope_formula = 1 - (1 - alpha) * (delta + n + g)
h = 1e-6
slope_numeric = (k_next(kstar + h) - k_next(kstar - h)) / (2 * h)
print(f"g'(k*) formula {slope_formula:.4f}, numeric {slope_numeric:.4f}")
print(f"the gap halves every {np.log(0.5) / np.log(slope_formula):.2f} periods")

fig, ax = plt.subplots(figsize=(6, 4))
for k0 in (0.25, 1.25, 3.5):
    ax.semilogy(np.abs(simulate(k0, 40) - kstar), label=f"$k_0 = {k0}$")
ax.set_xlabel("period"); ax.set_ylabel("$|k_t - k^*|$"); ax.legend()
plt.show()

# slower convergence when delta + n + g is small: g'(k*) closer to 1
slow = dict(delta=0.02)
x_slow, n_slow = fixed_point(lambda k: k_next(k, **slow), 0.25)
print(f"with delta = 0.02: g'(k*) = {1 - (1 - alpha) * (0.02 + n + g):.3f}, "
      f"{n_slow} iterations instead of {n_it}")

## Task 7: from notebook to module

The notebook was the lab bench. The code goes into a module so it can be tested and reused.

1. Put `k_next`, `steady_state` and `simulate` in `src/solow.py`.
2. Put this test in `tests/test_solow.py` and run `uv run pytest`:

```python
from solow import simulate, steady_state

def test_path_converges_to_steady_state():
    path = simulate(k0=0.25, T=200)
    assert abs(path[-1] - steady_state()) < 1e-6
```

3. Commit both files with a message that says what they do. Push.

That is M0. You have until the end of Week 3.